# Mini Project 1 — Analysis Notebook

**Your name:**  Nina Nguyen
**Dataset:**  'top_anime_100.csv'
**Date:**  Wed, May 6th

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** I'll be using the Jikan API (https://jikan.moe/), an unofficial REST API that pulls data from MyAnimeList. Specifically, I'll use the /top/anime endpoint to pull the top 100 anime by popularity, which returns each title's score, studio(s), genres, format, episode count, and air dates.

**Why this dataset:** As an avid anime watcher since childhood, I've always wondered whether factors like genre, animation studio, or decade of release influence how people ultimately rate a show. This also connects to my HCD interest in how creator choices shape audience reception, and whether aggregate ratings can reveal patterns that individual reviews can't.

**Three analytical questions:**

1. Among the 10 studios with the most titles in the top 100 anime, which has the highest average user score?
2. Among the top 100 anime, which 5 genres appear most frequently?
3. Among the top 100 anime, which decade of release (e.g., 2000s, 2010s, 2020s) has the highest average user score?

**What a practitioner would do with these findings:** With these findings, a practitioner could identify which studios, genres, and release periods are most associated with highly rated anime to guide content acquisition, production, and marketing decisions. These insights could also help streaming platforms improve recommendation systems and better target audience preferences. 

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [2]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_csv("top_anime_100.csv")
rows, cols = df.shape
print(f"Rows: {rows}")
print(f"Columns: {cols}")

Rows: 100
Columns: 59


In [3]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 59 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   mal_id                            100 non-null    int64  
 1   url                               100 non-null    str    
 2   approved                          100 non-null    bool   
 3   titles                            100 non-null    str    
 4   title                             100 non-null    str    
 5   title_english                     95 non-null     str    
 6   title_japanese                    100 non-null    str    
 7   title_synonyms                    100 non-null    str    
 8   type                              100 non-null    str    
 9   source                            100 non-null    str    
 10  episodes                          98 non-null     float64
 11  status                            100 non-null    str    
 12  airing              

In [4]:
# Summary statistics for numeric columns
df.describe()

,mal_id,episodes,score,scored_by,rank,popularity,members,favorites,year,trailer.youtube_id,...,trailer.images.small_image_url,trailer.images.medium_image_url,trailer.images.large_image_url,trailer.images.maximum_image_url,aired.prop.from.day,aired.prop.from.month,aired.prop.from.year,aired.prop.to.day,aired.prop.to.month,aired.prop.to.year
count,100.000000,98.000000,100.000000,1.000000e+02,99.000000,100.000000,1.000000e+02,100.000000,69.000000,0.0,...,0.0,0.0,0.0,0.0,100.000000,100.000000,100.000000,75.000000,75.000000,75.000000
mean,35619.390000,19.846939,8.794500,4.965350e+05,50.121212,917.430000,8.640355e+05,30097.850000,2016.536232,NaN,...,NaN,NaN,NaN,NaN,11.150000,6.050000,2016.970000,20.666667,6.666667,2016.960000
std,21128.468762,28.702653,0.144176,5.608381e+05,28.920818,1172.400234,8.656511e+05,49603.626748,8.780962,NaN,...,NaN,NaN,NaN,NaN,7.375054,3.562926,9.074835,9.099054,3.457594,8.538371
min,1.000000,1.000000,8.620000,2.658000e+03,1.000000,3.000000,1.073100e+04,81.000000,1980.000000,NaN,...,NaN,NaN,NaN,NaN,1.000000,1.000000,1980.000000,1.000000,1.000000,1981.000000
25%,16659.750000,7.750000,8.687500,1.112462e+05,25.500000,113.250000,2.526785e+05,2553.750000,2012.000000,NaN,...,NaN,NaN,NaN,NaN,6.000000,4.000000,2013.000000,15.000000,3.000000,2013.000000
50%,39690.000000,13.000000,8.750000,2.291865e+05,50.000000,539.000000,4.835015e+05,9177.000000,2019.000000,NaN,...,NaN,NaN,NaN,NaN,8.000000,6.500000,2020.000000,24.000000,6.000000,2019.000000
75%,53279.000000,24.000000,8.900000,8.271682e+05,74.500000,1113.500000,1.340763e+06,31923.500000,2023.000000,NaN,...,NaN,NaN,NaN,NaN,16.250000,10.000000,2024.000000,28.000000,9.000000,2023.500000
max,61952.000000,201.000000,9.270000,2.310051e+06,102.000000,7076.000000,3.679834e+06,251747.000000,2026.000000,NaN,...,NaN,NaN,NaN,NaN,30.000000,12.000000,2026.000000,31.000000,12.000000,2026.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

My dataset contains 100 rows and 59 columns. Each column represents a specific detail about an anime, including the number of episodes, average score, airing dates, and production studio. There are some missing entries for some rows, for example, "title_english" only has 95 out of the 100 animes.

My analysis will focus on the studios, score, genres, and year columns, since they are the most relevant to my analytical questions. However, 'year' only contains 69 entries, which may limit parts of the analysis involving release dates.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** Among the 10 studios with the most titles in the top 100 anime, which has the highest average user score?

In [7]:
# Retrieve the top 10 studios with the highest average user score,
# then sort them from highest to lowest score
df.groupby('studios')['score'].mean().sort_values(ascending=False).head(10)

studios
[{'mal_id': 287, 'type': 'anime', 'name': 'David Production', 'url': 'https://myanimelist.net/anime/producer/287/David_Production'}]                                                                                                                                              9.140000
[{'mal_id': 1269, 'type': 'anime', 'name': 'K-Factory', 'url': 'https://myanimelist.net/anime/producer/1269/K-Factory'}, {'mal_id': 2256, 'type': 'anime', 'name': 'Kitty Film Mitaka Studio', 'url': 'https://myanimelist.net/anime/producer/2256/Kitty_Film_Mitaka_Studio'}]    9.020000
[{'mal_id': 314, 'type': 'anime', 'name': 'White Fox', 'url': 'https://myanimelist.net/anime/producer/314/White_Fox'}]                                                                                                                                                            8.995000
[{'mal_id': 1258, 'type': 'anime', 'name': 'Bandai Namco Pictures', 'url': 'https://myanimelist.net/anime/producer/1258/Bandai_Namco_Pictures'}

In [8]:
# Reorganize the data above into a tidy table for readability, while omitting mal_id, 
# type, anime, name, and URL as these details are irrelevant to the question

import ast
import pandas as pd
from IPython.display import display

def studio_names(cell):
    if pd.isna(cell):
        return []
    try:
        lst = ast.literal_eval(cell)
    except (ValueError, SyntaxError):
        return []
    return [s["name"] for s in lst]

tmp = df.assign(studio_name=df["studios"].apply(studio_names)).explode("studio_name")

top10_by_score = (
    tmp.groupby("studio_name", as_index=False)
    .agg(mean_score=("score", "mean"), title_count=("mal_id", "count"))
    .sort_values("mean_score", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

display(top10_by_score)

,studio_name,mean_score,title_count
0,David Production,9.140000,1
1,K-Factory,9.020000,1
2,Kitty Film Mitaka Studio,9.020000,1
3,White Fox,8.995000,2
4,Bandai Namco Pictures,8.954000,5
5,TMS Entertainment,8.930000,1
6,Madhouse,8.894286,7
7,Sunrise,8.892857,7
8,TOHO animation STUDIO,8.880000,2
9,Bones,8.870000,3


**Note:**
After looking at the data in the above table, I realized that the suggested code was not correctly answering my question. My question asked, “Among the top 10 studios with the most titles...,” but it seems the Cursor agent only sorted studios by highest average user score rather than accounting for the number of titles produced by each studio.

In [5]:
# Your analysis for Question 1
# First find studios with the most titles, then compare average user score

import ast
import pandas as pd
from IPython.display import display

# Give me a simple list of studio names
def studio_names(cell):
    if pd.isna(cell):
        return []
    try:
        lst = ast.literal_eval(cell)
    except (ValueError, SyntaxError):
        return []
    return [s["name"] for s in lst]

# Turn that list into 1 row per studio, so I can count and average by studio
tmp = df.assign(studio_name=df["studios"].apply(studio_names)).explode("studio_name")

# Summarize every studio with average score and number of titles
studio_stats = (
    tmp.groupby("studio_name", as_index=False)
    .agg(mean_score=("score", "mean"), title_count=("mal_id", "count"))
)

# Collect the top 10 studios with the most titles and their average scores
top10_most_titles = (
    studio_stats.sort_values(["title_count", "mean_score"], ascending=[False, False])
    .head(10)
    .reset_index(drop=True)
)

display(top10_most_titles)

# Show the studio that has the highest average score, which answers my question
best_avg_in_top10 = top10_most_titles.sort_values("mean_score", ascending=False).head(1)
display(best_avg_in_top10)

,studio_name,mean_score,title_count
0,MAPPA,8.811250,8
1,Madhouse,8.894286,7
2,Sunrise,8.892857,7
3,Studio Pierrot,8.761429,7
4,Bandai Namco Pictures,8.954000,5
5,Kyoto Animation,8.822000,5
6,Shaft,8.790000,5
7,A-1 Pictures,8.760000,4
8,Studio Signpost,8.742500,4
9,Bones,8.870000,3


,studio_name,mean_score,title_count
4,Bandai Namco Pictures,8.954,5


**Interpretation:**  

**Out of the top 10 studios with the most titles in the top 100 anime, Bandai Namco Pictures has the highest average score of 8.954, with 5 titles represented.**

This result suggests that although some studios produce more top-ranked anime overall, Bandai Namco Pictures has the strongest average audience reception among the top 10 studios represented. This was somewhat surprising given that larger studios with more titles might be expected to dominate, and it would be interesting to further investigate whether specific genres, franchises, or release periods contributed to Bandai Namco Pictures’ high average score.

**Question 2:** *Among the top 100 anime, which 5 genres appear most frequently?*

In [6]:
# Your analysis for Question 2
import ast
from IPython.display import display

# Produce a clean list of genre names
def genre_names(cell):
    if pd.isna(cell):
        return []
    try:
        lst = ast.literal_eval(cell)
    except (ValueError, SyntaxError):
        return []
    return [g["name"] for g in lst]

# Tell me which 5 genres occur most often and how many times each occur
top5_genre_names = (
    df.assign(genre_name=df["genres"].apply(genre_names))
    .explode("genre_name")
    ["genre_name"]
    .value_counts()
    .head(5)
    .rename_axis("genre")
    .reset_index(name="count")
)

display(top5_genre_names)

,genre,count
0,Drama,47
1,Action,44
2,Adventure,22
3,Supernatural,19
4,Comedy,19


**Interpretation:**  

**The 5 genres that appear most frequently in the top 100 anime are: Drama (47), Action (44), Adventure (22), Supernatural (19), and Comedy (19).**

A practitioner could use this finding to better understand the types of stories and emotional experiences audiences are most drawn to when designing anime-related platforms, recommendation systems, or marketing strategies. These insights could help tailor user experiences around the genres that resonate most strongly with viewers.

**Question 3:** Among the top 100 anime, which decade of release (e.g., 2000s, 2010s, 2020s) has the highest average user score?

In [9]:
import pandas as pd
from IPython.display import display

# Keep anime with a known year in 1980-2026
period = df.dropna(subset=["year"]).loc[lambda d: (d["year"] >= 1980) & (d["year"] <= 2026)].copy()

# Bucket anime into 5 decade ranges
period["decade"] = pd.cut(
    period["year"],
    bins=[1979, 1989, 1999, 2009, 2019, 2026],
    labels=["1980-1989", "1990-1999", "2000-2009", "2010-2019", "2020-2026"],
    include_lowest=True,
    right=True,
)

# Compute average score and count per range, and sort by average score
score_by_decade = (
    period.groupby("decade", observed=True)
    .agg(mean_score=("score", "mean"), anime_count=("mal_id", "count"))
    .reset_index()
    .sort_values("mean_score", ascending=False)
)

display(score_by_decade)

# Report the top decade
winner = score_by_decade.iloc[0]
print(
    f"Highest average score: {winner['decade']} "
    f"(mean score {winner['mean_score']:.4f}, n={int(winner['anime_count'])})"
)

,decade,mean_score,anime_count
2,2000-2009,8.823000,10
3,2010-2019,8.820435,23
0,1980-1989,8.820000,1
4,2020-2026,8.782500,32
1,1990-1999,8.720000,3


Highest average score: 2000-2009 (mean score 8.8230, n=10)


**Note:** When I reviewed the table and added the counts, I realized they only summed to 69, even though there should have been a total of 100 anime. I then asked Cursor where the other 31 entries went, and it revealed that those anime did not have a year listed.

In [10]:
# Your analysis for Question 3
# Missing year -> "Unknown". Still excludes non-null years before 1980 or after 2026.

# Check through titles to see if they have a listed year between 1980 and 2026
has_year = df["year"].notna()
in_range = (df["year"] >= 1980) & (df["year"] <= 2026)

# For animes with known years, categorize them into decade range
bucketed = df.loc[has_year & in_range].copy()
bucketed["decade"] = pd.cut(
    bucketed["year"],
    bins=[1979, 1989, 1999, 2009, 2019, 2026],
    labels=["1980-1989", "1990-1999", "2000-2009", "2010-2019", "2020-2026"],
    include_lowest=True,
    right=True,
)

# Take anime with unlisted year and put them together in Unknown group
unknown_year = df.loc[~has_year].copy()
unknown_year["decade"] = "Unknown"

period_with_unknown = pd.concat([bucketed, unknown_year], ignore_index=True)

# Compute average score and count per range, and sort by average score
score_by_decade_with_unknown = (
    period_with_unknown.groupby("decade", observed=True)
    .agg(mean_score=("score", "mean"), anime_count=("mal_id", "count"))
    .reset_index()
    .sort_values("mean_score", ascending=False)
)

display(score_by_decade_with_unknown)

# Display decade with top average score
winner_u = score_by_decade_with_unknown.iloc[0]
print(
    f"Highest average score: {winner_u['decade']} "
    f"(mean score {winner_u['mean_score']:.4f}, n={int(winner_u['anime_count'])})"
)

,decade,mean_score,anime_count
2,2000-2009,8.823000,10
3,2010-2019,8.820435,23
0,1980-1989,8.820000,1
5,Unknown,8.784839,31
4,2020-2026,8.782500,32
1,1990-1999,8.720000,3


Highest average score: 2000-2009 (mean score 8.8230, n=10)


**Interpretation:**  

**The decade with the highest average score is 2000–2009, with an average score of 8.8230 across 10 anime titles.**

A practitioner could use this finding to highlight and recommend highly rated anime from the 2000s in streaming platforms, marketing campaigns, or curated collections. This finding is significant because it suggests that anime from this decade may have had a particularly strong impact on audiences, which could reflect trends in storytelling, animation style, or cultural influence during that time period.

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [17]:
import plotly.express as px

decade_labels = {
    "1980-1989": "1980's<br><sub>(1980-1989)</sub>",
    "1990-1999": "1990's<br><sub>(1990-1999)</sub>",
    "2000-2009": "2000's<br><sub>(2000-2009)</sub>",
    "2010-2019": "2010's<br><sub>(2010-2019)</sub>",
    "2020-2026": "2020's<br><sub>(2020-2026)</sub>",
    "Unknown": "Unknown",
}

chart_df = score_by_decade_with_unknown.copy()
chart_df["decade_label"] = chart_df["decade"].map(decade_labels)
chart_df = chart_df.sort_values("mean_score", ascending=True)
chart_df["bar_label"] = (
    chart_df["mean_score"].round(4).astype(str)
    + "  (" + chart_df["anime_count"].astype(int).astype(str) + " titles)"
)

fig = px.bar(
    chart_df,
    x="mean_score",
    y="decade_label",
    orientation="h",
    title="The 2000's Decade Has the Highest Average Score",
    labels={"decade_label": "Decade", "mean_score": "Average Score"},
    text="bar_label",
)

fig.update_xaxes(range=[8.6, 8.9])
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="Decade", xaxis_title="Average Score", margin=dict(l=120))
fig.show()

**Chart rationale:**  
I chose a horizontal bar chart because it is well suited for comparing a single numeric value — average user score — across several categories. The average scores are very close together (ranging from roughly 8.72 to 8.82), so a bar chart with a narrowed x-axis (starting at 8.6 instead of 0) makes those small but meaningful differences visually clear. A pie chart would not work here because the slices would appear nearly identical in size, and average scores are not parts of a whole.

The reader should take away that the 2000s decade edges out the other periods with the highest average score (8.823), though the 2010s and 1980s are extremely close behind. The chart also shows how many titles fall into each decade, highlighting that the 2020s have the most titles (32) but a slightly lower average, while the 2000s achieve the top average with only 10 titles.

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.